In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "5d733eab198ad58f23ceee3f1550014385366ece"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/5d733eab198ad58f23ceee3f1550014385366ece/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# GPUs, Devices, and Memory

So far every tensor in this book has lived in main memory and every computation
has run on the CPU. Training at any interesting scale runs on an accelerator,
almost always a CUDA GPU, and that changes two things at once. First, every
tensor and every parameter now has a *home*, a device, and operations only
combine tensors that live on the same one; placing data well is your job.
Second, the GPU's memory is small compared with main memory, typically tens of
gigabytes, and it is the resource you will exhaust first: the daily question of
a single-GPU builder is not "is my model correct?" but "does it fit?". This
section covers both skills: naming devices and placing tensors and models on
them, then measuring what actually fills GPU memory during training, trading
compute for memory when it does not fit, and keeping the device busy once it
does. None of the code requires a GPU to run; the helpers we define fall back
to the CPU, and the memory measurements simply report more interesting numbers
when a GPU is present.

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import time
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

## Devices

TensorFlow names devices with strings. `'/CPU:0'` is the CPU and it stands for
*all* physical CPU cores and all of main memory; `'/GPU:0'` is one specific
CUDA card and its own memory, and `'/GPU:1'` is the second card. `tf.device`
turns a name into a *scope*: operations executed inside it run on, and
allocate their results on, the named device. To see what your machine has,
run `nvidia-smi` in a shell: it lists every card, its memory, and what is
currently running on it. We wrap device construction in two helpers that the
rest of the book uses.

In [ ]:
def cpu():
    """Get the CPU device."""
    return tf.device('/CPU:0')
def gpu(i=0):
    """Get a GPU device."""
    return tf.device(f'/GPU:{i}')
cpu(), gpu(), gpu(1)

Note that constructing a device scope is free and always succeeds, whether or
not the hardware exists; the name is checked only when an operation runs
inside the scope. Even then, TensorFlow's default *soft placement* falls back
to the CPU rather than raising when the requested device is missing, which is
what lets `gpu(1)` print happily on a laptop. The flip side of that tolerance
is that a mistyped device name never crashes; check `x.device` when placement
matters.

We can query how many GPUs are actually available.

In [ ]:
def num_gpus():
    """Get the number of available GPUs."""
    return len(tf.config.experimental.list_physical_devices('GPU'))
num_gpus()

Now we define two convenient functions that allow us
to run code even if the requested GPUs do not exist.

In [ ]:
def try_gpu(i=0):
    """Return gpu(i) if exists, otherwise return cpu()."""
    if num_gpus() >= i + 1:
        return gpu(i)
    return cpu()

def try_all_gpus():
    """Return all available GPUs, or [cpu(),] if no GPU exists."""
    devices = [gpu(i) for i in range(num_gpus())]
    return devices if devices else [cpu()]

try_gpu(), try_gpu(10), try_all_gpus()

CUDA is not the only accelerator TensorFlow understands: the same device
strings name TPUs (`'/TPU:0'`) on a TPU host, and the `tensorflow-metal`
plugin exposes an Apple-silicon GPU as `'/GPU:0'`;
`tf.config.list_physical_devices()` with no argument lists everything the
runtime can see. We keep our own helpers anyway: they give all four of the
book's implementations one vocabulary, and they degrade to the CPU instead of
failing, which is exactly what lets this book's code run unchanged on a
laptop. The book standardizes on CPU plus CUDA; everything below transfers to
the other backends with little more than a renamed device string.

## Tensors, Models, and Devices

By default, tensors are created on the first GPU whenever one exists and on
the CPU otherwise: eager TensorFlow places every new tensor on `'/GPU:0'` if
it can. We can query where a tensor lives.

In [ ]:
x = tf.constant([1.0, 2.0, 3.0])
x.device

To place a tensor somewhere specific, create it inside a device scope:
everything executed under `with try_gpu():` allocates its results on that
device. Creating data directly on the target device is better than creating
it on the CPU and moving it: the tensor then never occupies main memory or
crosses the bus at all.

In [ ]:
with try_gpu():
    X = tf.ones((2, 3))
X

On a machine with two GPUs we can put a second tensor on the second card
(on your machine, `try_gpu(1)` substitutes whatever is available).

In [ ]:
with try_gpu(1):
    Y = tf.random.uniform((2, 3))
Y

### Copying Between Devices

Whenever we operate on multiple tensors, they should be on the same device.
TensorFlow's eager *device policy* defaults to silent, so if `X` sits on the
first GPU and `Y` on the
second, `X + Y` does not raise. TensorFlow copies one operand across and
returns an answer. This hides a slow bus transfer inside an innocent-looking
`+`, and nothing in your code marks the line that pays for it.
`tf.config.experimental.set_device_policy('explicit')` opts into strictness,
turning silent copies into errors. Either way the discipline is the same:
copy explicitly, as in the figure, and then add. The explicit copy
in TensorFlow is `tf.identity` inside a device scope.

![X lives on GPU 0 and Y on GPU 1; X.to(gpu(1)) makes a copy Z on GPU 1 (dashed), and Y + Z then runs entirely on GPU 1.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-copyto.svg)

In [ ]:
with try_gpu(1):
    Z = tf.identity(X)
print(X)
print(Z)

Now that `Z` and `Y` live on the same device, we can add them.

In [ ]:
Y + Z

What if `Z` already lives on the target device? `tf.identity` under a scope
naming that same device stays put: no bus transfer happens, so the defensive
copy costs a kernel launch and nothing more. In everyday TensorFlow you will
rarely write even that much, since the silent policy fetches remote operands
on demand; the point of spelling out the copy is to make the one expensive
transfer visible in your code instead of buried in an operator.

The reason for keeping every copy explicit is the cost model. A
modern GPU multiplies matrices hundreds of times faster than it can receive
data over the PCIe bus, so a transfer in the wrong place can erase the
speedup you bought the GPU for. The discipline that follows is simple: move data to the device
once, at the boundary of the computation, and keep everything inside the
training loop on one device. Many small transfers are worse than one big one,
and a transfer hidden in an inner loop is worst of all.

### Models on a Device

A Keras model has no device argument and no move method: layers create their
variables on the first call, and the variables live wherever that first call
ran. So we run the first forward pass inside a device scope, and the
parameters stay on that device from then on.

In [ ]:
net = tf.keras.Sequential([tf.keras.layers.Dense(1)])
with try_gpu():
    Y_hat = net(X)  # The first call creates the variables inside the scope
Y_hat

The input arrived on the device, the parameters live on the device, so the
output is computed and stored there too. Let's confirm where the parameters
ended up.

In [ ]:
net.layers[0].kernel.value.device

Device hygiene is short in TensorFlow because placement follows execution. A
fresh tensor made during the forward pass is created where the surrounding
ops run, and a Keras optimizer creates its slot variables at the first
`apply_gradients`, next to the variables they update, so ordering takes care
of itself. The one rule to remember is the first-call rule above: run the
variable-building forward pass under the device scope you mean, because
moving the variables afterwards means rebuilding the model.

## GPU Memory

Here is a puzzle that every TensorFlow user hits in their first week. You
create one small tensor, yet `nvidia-smi` shows the card nearly full; is that
a leak? It is not, and the explanation is the right mental model for
everything else in this section. Requesting memory from the CUDA driver is
slow, so on first use TensorFlow maps nearly all of the GPU's memory in one
grab and hands out pieces of it from its own allocator;
`tf.config.experimental.set_memory_growth(gpu, True)`, called before the
first computation, switches to growing the reservation on demand instead.
`nvidia-smi` sees the process from the outside, so it reports the whole
reservation no matter how little of it your tensors occupy. The real
accounting lives inside: `tf.config.experimental.get_memory_info('GPU:0')`
returns a dictionary whose `'current'` entry counts live tensors and whose
`'peak'` entry is their high-water mark, the same nesting of views that
the figure sketches. There is no such allocator for the CPU,
and the call raises on `'CPU:0'`.

![The three memory views nest: nvidia-smi's driver allocation contains PyTorch's reserved cache, which contains the live tensors that memory_allocated() counts.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-allocator.svg)

In [ ]:
if num_gpus() > 0:
    def report(tag):
        info = tf.config.experimental.get_memory_info('GPU:0')
        print(f'{tag}: {info["current"] / 2**20:7.1f} MiB in use, '
              f'{info["peak"] / 2**20:7.1f} MiB peak')
    with gpu():
        big = tf.zeros((256, 1024, 1024))  # 1 GiB of float32
    report('after creating big')
    del big
    report('after deleting it ')
else:
    print('No GPU: get_memory_info() tracks only GPU and TPU allocators.')

On a GPU, `'current'` falls by a gibibyte the moment the last reference dies,
while `nvidia-smi` does not move at all: the block returns to TensorFlow's
allocator pool, not to the driver. There is no `empty_cache` equivalent; the
process keeps its reservation until it exits. If another process must share
the card, enable memory growth, or set a hard cap with
`tf.config.set_logical_device_configuration`, before the first computation.

### What Fills Memory During Training

In that section we did the bookkeeping on paper: a model with
$N$ parameters costs $4N$ bytes for float32 weights, another $4N$ for their
gradients, and $8N$ for Adam's two moment estimates, roughly $16N$ before any
data arrives. The remaining term, the *activations*, is different in kind:
backpropagation must remember the intermediate results of the forward pass in
order to compute gradients, and their size scales with the batch size and with
the width of every layer, while the $16N$ does not.

We can watch each term arrive by reading `'current'` at four points of a
single training step. The activations are visible as a plateau for the same
reason as in PyTorch: `tf.GradientTape` records the forward pass and holds
the intermediate results until we ask it for gradients.

In [ ]:
if num_gpus() > 0:
    def in_use():
        info = tf.config.experimental.get_memory_info('GPU:0')
        return f'{info["current"] / 2**20:7.1f} MiB'
    with gpu():
        net = tf.keras.Sequential(
            [tf.keras.layers.Dense(4096, activation='relu'),
             tf.keras.layers.Dense(4096, activation='relu'),
             tf.keras.layers.Dense(4096, activation='relu'),
             tf.keras.layers.Dense(10)])
        X = tf.random.normal((4096, 1024))
        y = tf.random.uniform((4096,), 0, 10, dtype=tf.int32)
        net(X)  # The first call creates the variables
    opt = tf.keras.optimizers.Adam()
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    print('weights           ', in_use())
    with tf.GradientTape() as tape:
        loss = loss_fn(y, net(X, training=True))
    print('+ activations     ', in_use())
    grads = tape.gradient(loss, net.trainable_variables)
    print('+ gradients       ', in_use())
    opt.apply_gradients(zip(grads, net.trainable_variables))
    print('+ optimizer state ', in_use())
else:
    print('No GPU: run this cell on a GPU to see the four plateaus.')

The four readings map onto the accounting. The first is the weights alone
(about $4N$ bytes, plus the input batch). The second jumps by the size of the
activations the tape is holding — here roughly a third more than the
weights; a larger batch or longer sequence scales this term while the
weights stay put. The third *falls* back: `tape.gradient` consumes the recorded
forward pass, frees it, and leaves behind gradients exactly the size of the
weights. The fourth adds $8N$ at the first `apply_gradients`, when Adam
creates its slot variables. From then on the loop cycles between plateaus two
and three; the weights, gradients, and optimizer state are permanent
residents. The practical consequence: when you are out of memory, the knob
that works immediately is the batch size, because it scales the one term the
model architecture does not fix. If a *growing* staircase appears across
steps instead of this steady cycle, look for tensors kept alive between
iterations, classically a Python list accumulating the loss tensor itself
rather than `float(loss)`, or a persistent `GradientTape` that never dies.

### Trading Compute for Memory: Activation Checkpointing

The activation term has a second knob. Backpropagation stores every
intermediate result only to read each one exactly once, on the way back.
*Activation checkpointing* [@Chen.Xu.Zhang.ea.2016] refuses to store
them: it keeps only the inputs to selected segments of the network, and during
the backward pass reruns each segment's forward computation to regenerate the
activations it needs. For a stack of $N$ equally sized layers, dividing the
stack into segments of about $\sqrt{N}$ layers reduces stored activations from
$O(N)$ to $O(\sqrt{N})$. The price is roughly one extra forward pass, often
30--40% more compute per step. the figure sketches
this schedule.
The trade pays off exactly where deep learning spends most of its time: deep
stacks of identical blocks, such as the residual stack we assembled in
that section and every Transformer you will meet later.
We rebuild a compact version of that block here.

![Standard backpropagation stores every activation (top); checkpointing keeps only segment-boundary activations and recomputes the rest during the backward pass (bottom), trading roughly 1.3x compute for O(sqrt(N)) instead of O(N) activation memory.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-activation-checkpoint.svg)

In [ ]:
class ResidualBlock(tf.keras.Model):  # As in :numref:`sec_model_construction`
    def __init__(self, num_hiddens):
        super().__init__()
        self.body = tf.keras.Sequential(
            [tf.keras.layers.Dense(num_hiddens, activation='relu'),
             tf.keras.layers.Dense(num_hiddens, activation='relu')])

    def call(self, X):
        return X + self.body(X)

def run_stack(blocks, X, use_checkpoint=False, segment_size=4):
    for start in range(0, len(blocks), segment_size):
        segment_blocks = blocks[start:start + segment_size]
        def segment(X):
            for blk in segment_blocks:
                X = blk(X)
            return X
        f = tf.recompute_grad(segment) if use_checkpoint else segment
        X = f(X)
    return X

`tf.recompute_grad(blk)` returns a function that behaves exactly like `blk`
on the forward pass but tells the tape to store only the inputs, rerunning
`blk` during the backward pass to regenerate what it needs. One caveat:
unlike its PyTorch counterpart it does not snapshot random-number state, so
keep checkpointed segments deterministic (or drive their randomness with
stateless seeds); a classic dropout layer inside the segment could resample
on recomputation.

Before measuring memory, we should verify the claim that nothing about the
result changes: the gradients through a checkpointed stack must equal the
ordinary ones.

In [ ]:
tf.keras.utils.set_random_seed(0)
blocks = [ResidualBlock(64) for _ in range(4)]
X = tf.random.normal((32, 64))

def stack_grads(use_checkpoint):
    with tf.GradientTape() as tape:
        loss = tf.reduce_sum(run_stack(blocks, X, use_checkpoint))
    params = [v for blk in blocks for v in blk.trainable_variables]
    return tape.gradient(loss, params)

for g, g_ckpt in zip(stack_grads(False), stack_grads(True)):
    assert np.allclose(g, g_ckpt)
print('checkpointed gradients match the ordinary ones')

That check runs anywhere, CPU included. The memory effect needs a GPU and a
stack deep enough for the difference to be unambiguous: sixteen blocks of
width 1024 at batch size 8192.

In [ ]:
if num_gpus() > 0:
    with gpu():
        blocks = [ResidualBlock(1024) for _ in range(16)]
        X = tf.random.normal((8192, 1024))
        run_stack(blocks, X)  # Create the variables before measuring
    params = [v for blk in blocks for v in blk.trainable_variables]
    for use_checkpoint in (False, True):
        tf.config.experimental.reset_memory_stats('GPU:0')
        t = time.time()
        with tf.GradientTape() as tape:
            loss = tf.reduce_sum(run_stack(blocks, X, use_checkpoint))
        grads = tape.gradient(loss, params)
        tf.test.experimental.sync_devices()
        peak = tf.config.experimental.get_memory_info('GPU:0')['peak'] / 2**20
        print(f'checkpointing={use_checkpoint!s:5}  peak {peak:6.0f} MiB, '
              f'{time.time() - t:.2f} sec')
else:
    print('No GPU: peak-memory comparison needs a GPU.')

Without checkpointing, the peak carries the activations of all sixteen blocks
at once; with four-block segments, it carries the four segment inputs plus the
recomputed activations of one segment, at the cost of a slower step. When a
model almost fits, this
trade is the difference between training and not training, which is why large
Transformer training runs use it as a matter of course.

## Don't Break the Pipeline

The GPU runs ahead of Python. When you write `B = A @ A` on a GPU tensor,
eager TensorFlow does not wait for the multiplication: it enqueues the kernel
on the device's stream and returns a handle immediately, and Python races on
to enqueue the next operation (inside a `@tf.function`, whole traced graphs
are handed to the runtime the same way). This asynchrony is where much of the
speed comes from, because the CPU can prepare work while the GPU crunches. It
also means that timing or logging naively measures nothing, or worse, stalls
everything. Eager ops on the CPU backend run synchronously, so seeing the gap
between *queueing* work and the work *finishing* needs a GPU.

In [ ]:
if num_gpus() > 0:
    with gpu():
        A = tf.random.normal((4096, 4096))
    (A @ A).numpy()  # Warm up
    t = time.time()
    for _ in range(32):
        B = A @ A
    print(f'time to queue 32 matrix products: {time.time() - t:.4f} sec')
    tf.test.experimental.sync_devices()
    print(f'time until they all finished:     {time.time() - t:.4f} sec')
else:
    print('No GPU: TensorFlow executes CPU ops synchronously.')

Python returned from the loop in a few milliseconds; the products were still
running. Any operation that needs a concrete value on the host forces a
*synchronization point*: `.numpy()`, `float(loss)`, `print`, an `if` on a
tensor's value. Each one makes Python block until the queue drains, and the
device then sits idle until Python catches up
(`tf.test.experimental.sync_devices()` is the explicit version that accurate
timings need). A `print(float(loss))` in the inner loop can serialize the
whole pipeline this way, once per step, as the figure lays
out. The fix is not to give up monitoring but to move it off the hot path:
keep running statistics on the device and transfer them once per epoch, or
hand values to a background consumer, which is exactly what our
`ProgressBoard` from that section does when the `Trainer` plots
the loss.

![Python queues kernels k1 through k4 and races ahead while the GPU works through them serially; loss.item() forces a synchronization point where the CPU blocks until the GPU drains its queue, then both resume.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-async-queue.svg)

The same overlap idea applies to getting data *onto* the device, and in
TensorFlow it belongs to the input pipeline rather than to a per-tensor flag:
ending a `tf.data` pipeline with `.prefetch(tf.data.AUTOTUNE)` keeps the next
batches being prepared in the background while the device computes, and
`tf.data.experimental.prefetch_to_device` stages them on the GPU ahead of
use. The book's data loaders are `tf.data.Dataset`s, so this section's
overlap discipline reduces to one appended call. The full treatment of
asynchrony, streams, and multi-device parallelism is in
that section.

## The Trainer, Now with Devices

In that section the `Trainer` accepted a `num_gpus` argument and
ignored it. We can now redeem that promise with everything this section
taught: the model's variables are created on the device by a first forward
pass (which the `Trainer` already runs before training, when it compiles its
step functions), and every batch is re-materialized on the device per step,
at the boundary of the computation.

In [ ]:
@d2l.add_to_class(d2l.Trainer)
def __init__(self, max_epochs, num_gpus=0, gradient_clip_val=0):
    self.save_hyperparameters()
    self.gpus = [d2l.gpu(i) for i in range(min(num_gpus, d2l.num_gpus()))]

@d2l.add_to_class(d2l.Trainer)
def prepare_batch(self, batch):
    if self.gpus:
        # tf.data.Dataset emits batches on CPU. Re-wrap them inside the
        # GPU device context so subsequent ops keep their inputs on-device
        # rather than incurring an implicit copy each step.
        with self.gpus[0]:
            batch = [tf.identity(a) for a in batch]
    return batch

The `min(num_gpus, d2l.num_gpus())` makes the request a ceiling rather than a
demand: on a machine with no GPU, `self.gpus` is empty, `prepare_batch` is a
no-op, and training proceeds on the CPU. The explicit `tf.identity` inside
the device scope matters because `tf.data` pipelines emit their batches on
the CPU: without it, every op that touches the batch pays the silent
host-to-device copy this section warned about, once per step. There is no
`prepare_model` patch to write, since Keras variables are created on the
first call and the `Trainer` runs its compile-time forward pass through
`prepare_batch`, so the weights land on the configured GPU automatically.
Every `Trainer(num_gpus=1)` you see from the next chapter onward relies on
this fallback, which is how one codebase serves both the laptop you are
reading on and a GPU server. As a capstone we train a classifier built from
this chapter's residual blocks on Fashion-MNIST; the memory budget of the
run is exactly the four-plateau accounting from above.

In [ ]:
class ResMLPClassifier(d2l.Classifier):
    def __init__(self, num_hiddens=256, num_blocks=2, lr=0.1):
        super().__init__()
        self.save_hyperparameters()
        self.net = tf.keras.Sequential(
            [tf.keras.layers.Flatten(),
             tf.keras.layers.Dense(num_hiddens, activation='relu'),
             *[ResidualBlock(num_hiddens) for _ in range(num_blocks)],
             tf.keras.layers.Dense(10)])

trainer = d2l.Trainer(max_epochs=1, num_gpus=1)
trainer.fit(ResMLPClassifier(), d2l.FashionMNIST(batch_size=256))

## Summary

Every tensor and every Keras variable lives on a device, and TensorFlow's
silent device policy will copy across devices for you, so keeping copies
explicit and at the boundary of the training loop is a discipline here rather
than an enforced rule; it matters just as much, because the hidden transfer
is just as slow. GPU memory during training holds four things: weights,
gradients, optimizer state (fixed by the model and optimizer), and the
activations the `GradientTape` records (scaling with batch size);
`get_memory_info` tracks live tensors inside the reservation TensorFlow maps
at startup, while `nvidia-smi` shows the whole reservation, so the two
disagree by design. Activation checkpointing (`tf.recompute_grad`) recomputes
instead of stores, trading roughly a third more compute for activation
memory. On a GPU, eager kernels run asynchronously ahead of Python:
`.numpy()`, `float()`, and `print` are synchronization points that stall the
pipeline, and `.prefetch(tf.data.AUTOTUNE)` is where input transfer overlaps
compute. The `Trainer` encodes the placement discipline: variables created on
the device at the compile-time forward pass, batches re-wrapped per step,
graceful CPU fallback when no GPU exists.

## Exercises

1. Using the accounting model of this section, predict the peak memory of the
   four-plateau cell at batch sizes 64, 256, and 1024, then measure with
   your framework's peak-memory counter. Where does the prediction break down,
   and what did it omit?
1. Increase the batch size in the checkpointing comparison until the
   non-checkpointed run raises an out-of-memory error. How much further can
   the checkpointed run go before it does too? Explain the ratio using the
   sizes of what each variant stores.
1. Time one epoch of the capstone training run with `print(loss.item())`
   after every step, and again printing only once per epoch. Explain the
   difference in terms of synchronization points.
1. If you have two GPUs, time 1000 matrix products of two $4096 \times 4096$
   matrices executed on one GPU, then 500 on each of two GPUs issued from the
   same loop. You should see almost linear scaling; explain why Python can
   drive both cards at once. Guard the experiment with `num_gpus() >= 2`.